# 03 · Iteration & Generators

Set/dict comprehensions, generator expressions, `yield`, the iterator protocol (`__iter__`/`__next__`) by hand, and `yield from`. Lazy evaluation is the through-line.

### 3.1 — Set and dict comprehensions

You know list comprehensions. The same syntax builds sets `{...}` and dicts `{k: v ...}`.

Return a dict mapping each word to its length, but only for words longer than 2 characters.

```
lengths(["a", "cat", "hi", "dog"]) → {"cat": 3, "dog": 3}
```

In [1]:
def lengths(words):
    return {word: len(word) for word in words if len(word) > 2}

# Tests
assert lengths(["a", "cat", "hi", "dog"]) == {"cat": 3, "dog": 3}
assert lengths(["ab"]) == {}
print("3.1 passed")

3.1 passed


### 3.2 — Generator expressions

A generator expression looks like a comprehension with `()` instead of `[]`. It's **lazy** — it 
produces values one at a time instead of building a whole list in memory. Perfect for feeding 
into `sum`, `any`, `all`, `max`.

Using a generator expression inside `sum`, return the sum of squares of a list — without building 
an intermediate list.

```
sum_of_squares([1,2,3]) → 14
```

In [2]:
def sum_of_squares(nums):
    return sum((num ** 2 for num in nums))

# Tests
assert sum_of_squares([1, 2, 3]) == 14
assert sum_of_squares([]) == 0
print("3.2 passed")

3.2 passed


### 3.3 — Writing a generator with `yield`

A function containing `yield` is a **generator**: calling it returns a generator object that produces 
values lazily, pausing and resuming at each `yield`. State is preserved between yields.

Write `count_up_to(n)` that yields `1, 2, ..., n` one at a time.

```
list(count_up_to(4)) → [1, 2, 3, 4]
```

In [8]:
def count_up_to(n):
    for i in range(1, n+1):
        yield i

# Tests
assert list(count_up_to(4)) == [1, 2, 3, 4]
assert list(count_up_to(1)) == [1]
assert list(count_up_to(0)) == []
gen = count_up_to(3)
assert next(gen) == 1       # produces lazily, one at a time
assert next(gen) == 2
print("3.3 passed")

3.3 passed


### 3.4 — A stateful generator

Generators shine when state accrues. Write `running_total(nums)` that yields the cumulative sum 
after each element — lazily.

```
list(running_total([1,2,3,4])) → [1, 3, 6, 10]
```

(Same result as your `accumulate` from the first set — but lazy, one value at a time.)

In [12]:
def running_total(nums):
    total = 0

    for num in nums:
        total += num
        yield total

# Tests
assert list(running_total([1, 2, 3, 4])) == [1, 3, 6, 10]
assert list(running_total([])) == []
print("3.4 passed")

3.4 passed


### 3.5 — The iterator protocol by hand

`for` works on anything implementing the **iterator protocol**: `__iter__` (returns an iterator) and 
`__next__` (returns the next value or raises `StopIteration`). Generators implement this for you; 
here you do it manually.

Write a `Countdown` class so that iterating it counts down from `start` to `1`.

```
list(Countdown(3)) → [3, 2, 1]
```

In [13]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        self.current = self.start
        return self

    def __next__(self):
        if self.current < 1:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

# Tests
assert list(Countdown(3)) == [3, 2, 1]
assert list(Countdown(1)) == [1]
assert list(Countdown(0)) == []
print("3.5 passed")

3.5 passed


In [20]:
class Fibonacci:
    def __init__(self, n):
        self.n = n          # how many numbers to produce

    def __iter__(self):
        self.counter = 0
        self.last = 0
        self.current = 1
        return self

    def __next__(self):
        if self.counter >= self.n:
            raise StopIteration
        self.counter += 1
        value = self.current
        self.last, self.current = self.current, self.last + self.current
        return value

# Tests
assert list(Fibonacci(5)) == [1, 1, 2, 3, 5]
assert list(Fibonacci(1)) == [1]
assert list(Fibonacci(0)) == []
assert list(Fibonacci(7)) == [1, 1, 2, 3, 5, 8, 13]
gen = iter(Fibonacci(3))
assert next(gen) == 1
assert next(gen) == 1
assert next(gen) == 2
print("bonus passed")

bonus passed


In [23]:
class Stepper:
    def __init__(self, items, step):
        self.items = items
        self.step = step

    def __iter__(self):
        self.current_index = 0
        return self

    def __next__(self):
        if self.current_index >= len(self.items):
            raise StopIteration
        value = self.items[self.current_index]
        self.current_index += self.step
        return value

# Tests
assert list(Stepper([10, 20, 30, 40, 50], 2)) == [10, 30, 50]   # indices 0, 2, 4
assert list(Stepper([1, 2, 3, 4], 1)) == [1, 2, 3, 4]            # every element
assert list(Stepper([1, 2, 3], 5)) == [1]                        # only index 0, then past end
assert list(Stepper([], 2)) == []                                # nothing to yield
print("stepper passed")

stepper passed


### 3.6 — `yield from`

`yield from <iterable>` yields every value from a sub-iterable — cleaner than a manual re-yield loop. 
Write `flatten_gen(nested)` that yields each element of a list of lists, in order, using `yield from`.

```
list(flatten_gen([[1,2],[3],[4,5]])) → [1, 2, 3, 4, 5]
```

In [24]:
def flatten_gen(nested):
    for sublist in nested:
        yield from sublist

# Tests
assert list(flatten_gen([[1, 2], [3], [4, 5]])) == [1, 2, 3, 4, 5]
assert list(flatten_gen([])) == []
print("3.6 passed")

3.6 passed
